# Desafio Wellbe — ETL

Extração, tratamento e carga de `data/dados.csv` em MySQL.

In [1]:
import re
import pandas as pd

RAW_PATH = "../data/dados.csv"

raw = pd.read_csv(RAW_PATH, skiprows=13, encoding="utf-8")

colunas_esperadas = {
    "Código": "codigo",
    "Custo do afastamento": "custo_raw",
    "Identificação": "identificacao_raw",
    "Funcionário": "funcionario_raw",
    "Departamento": "departamento_raw",
    "Data do Atestado": "data_raw",
    "Especialidade": "especialidade_raw",
    "Motivo": "motivo_raw",
    "Líder": "lider_raw",
}
assert set(colunas_esperadas.keys()) <= set(raw.columns), (
    f"Header inesperado: {raw.columns.tolist()}"
)
raw = raw.rename(columns=colunas_esperadas)

print(raw.shape)
raw.head()

(91, 9)


,codigo,custo_raw,identificacao_raw,funcionario_raw,departamento_raw,data_raw,especialidade_raw,motivo_raw,lider_raw
0,1036743,"20,4",NaN,Anonimo 101,Gerente,29/05/2019,Exames,Exames,Sim
1,1036742,23,NaN,Anonimo 1,ASSISTENTE DE IMPLANTACAO,23/05/2019,Neurologia pediátrica,Acompanhamento familiar,NaN
2,1036741,"22,8",NaN,Anonimo 1,ASSISTENTE DE IMPLANTACAO,17/05/2019,NaN,Consulta médica,NaN
3,1036740,–-,NaN,Anonimo 1,ASSISTENTE DE IMPLANTACAO,16/05/2019,Exames,Exames,NaN
4,1036739,22,NaN,Anonimo 1,ASSISTENTE DE IMPLANTACAO,06/05/2019,NaN,Consulta médica,NaN


In [2]:
def parse_custo(value):
    if pd.isna(value):
        return 0.0
    text = str(value).strip().replace(",", ".")
    if re.fullmatch(r"-?\d+(\.\d+)?", text):
        return float(text)
    return 0.0


def parse_data(value):
    if pd.isna(value) or str(value).strip() == "":
        return pd.NaT
    return pd.to_datetime(str(value).strip(), format="%d/%m/%Y", errors="coerce")


# Sanidade das regras de tratamento (DADOS.md)
assert parse_custo("20,4") == 20.4
assert parse_custo("--") == 0.0
assert parse_custo("–-") == 0.0
assert parse_custo(float("nan")) == 0.0
assert parse_custo("9215") == 9215.0  # valor bruto quebrado, tratado literalmente
assert parse_data("29/05/2019") == pd.Timestamp("2019-05-29")
assert pd.isna(parse_data(""))
assert pd.isna(parse_data(float("nan")))
print("helpers ok")

helpers ok


In [3]:
def transform(raw_df):
    departamentos = {}
    funcionarios = {}
    atestados = []
    lider_atual = None

    def dep_id(nome_raw):
        nome = str(nome_raw).strip() if pd.notna(nome_raw) else ""
        if nome == "":
            return None
        if nome not in departamentos:
            departamentos[nome] = len(departamentos) + 1
        return departamentos[nome]

    def func_id(nome_raw, eh_lider_linha):
        nome = str(nome_raw).strip()
        if nome not in funcionarios:
            funcionarios[nome] = {"id": len(funcionarios) + 1, "eh_lider": False}
        if eh_lider_linha:
            funcionarios[nome]["eh_lider"] = True
        return funcionarios[nome]["id"]

    for _, row in raw_df.iterrows():
        nome_funcionario = row["funcionario_raw"]
        if pd.isna(nome_funcionario) or str(nome_funcionario).strip() == "":
            continue

        eh_lider_linha = str(row["lider_raw"]).strip().lower() == "sim"
        funcionario_id = func_id(nome_funcionario, eh_lider_linha)

        if eh_lider_linha:
            lider_atual = str(nome_funcionario).strip()

        data = parse_data(row["data_raw"])
        if pd.isna(data):
            continue

        departamento_id = dep_id(row["departamento_raw"])
        lider_funcionario_id = (
            funcionarios[lider_atual]["id"] if lider_atual is not None else None
        )

        atestados.append({
            "id": int(row["codigo"]),
            "funcionario_id": funcionario_id,
            "departamento_id": departamento_id,
            "lider_funcionario_id": lider_funcionario_id,
            "data_atestado": data.date(),
            "especialidade": str(row["especialidade_raw"]).strip() if pd.notna(row["especialidade_raw"]) else "",
            "motivo": str(row["motivo_raw"]).strip() if pd.notna(row["motivo_raw"]) else "",
            "custo_afastamento": parse_custo(row["custo_raw"]),
        })

    return departamentos, funcionarios, atestados


departamentos, funcionarios, atestados = transform(raw)

departamentos_df = pd.DataFrame(
    [{"id": v, "nome": k} for k, v in departamentos.items()]
).sort_values("id").reset_index(drop=True)

funcionarios_df = pd.DataFrame(
    [{"id": v["id"], "nome": k, "identificacao": "", "eh_lider": v["eh_lider"]} for k, v in funcionarios.items()]
).sort_values("id").reset_index(drop=True)

atestados_df = pd.DataFrame(atestados)

print(departamentos_df.shape, funcionarios_df.shape, atestados_df.shape)
atestados_df.head()

(20, 2) (28, 4) (90, 8)


,id,funcionario_id,departamento_id,lider_funcionario_id,data_atestado,especialidade,motivo,custo_afastamento
0,1036743,1,1.0,1,2019-05-29,Exames,Exames,20.4
1,1036742,2,2.0,1,2019-05-23,Neurologia pediátrica,Acompanhamento familiar,23.0
2,1036741,2,2.0,1,2019-05-17,,Consulta médica,22.8
3,1036740,2,2.0,1,2019-05-16,Exames,Exames,0.0
4,1036739,2,2.0,1,2019-05-06,,Consulta médica,22.0


In [4]:
# Validações de integridade (sem suíte pytest formal, por escopo do desafio)
assert atestados_df["data_atestado"].isna().sum() == 0, "há atestado sem data válida"

nomes_unicos_csv = raw["funcionario_raw"].dropna().apply(lambda x: str(x).strip())
nomes_unicos_csv = nomes_unicos_csv[nomes_unicos_csv != ""].nunique()
assert len(funcionarios_df) == nomes_unicos_csv, (
    f"funcionarios_df tem {len(funcionarios_df)} linhas, CSV tem {nomes_unicos_csv} nomes únicos"
)

data_parsed = raw["data_raw"].apply(parse_data)
custo_esperado = raw.loc[data_parsed.notna(), "custo_raw"].apply(parse_custo).sum()
custo_carregado = atestados_df["custo_afastamento"].sum()
assert abs(custo_esperado - custo_carregado) < 0.01, (
    f"custo esperado {custo_esperado} != custo carregado {custo_carregado}"
)

print("validações ok")

validações ok
